In [1]:
from BinLattice import *
from Lattice import *
from LatticeUtils import *
from DiscForm import *
from IntVectors import *
from Vinberg import *
from FPSearch import *
from VSearch import *
from Allcock import *
from Circle import *
from Commons import *
import fp_search_cpp
import vsearch_cpp

In [7]:
M = Leech_lat()
A, T = M.lll(transform=True)
FPS = fp_search_cpp.FPSearch(np.array(A, dtype=float), np.zeros(M.rank, dtype=float), 0, 4.5)
vecs = FPS.search_all()
print(len(vecs))

196560


In [14]:
def cusp_walls(L: Lattice, v: IMat, base: IMat) -> List[IMat]:
    if L.signature[0] != 1:
        raise ValueError("The lattice must be of signature (1, n)")
    if L.square(v) != 0:
        raise ValueError("The vector v must be isotropic")
    if L.square(base) <= 0:
        raise ValueError("The base vector must be positive")
    v = imat(v) // math.gcd(*v.tolist())
    if L.product(v, base) < 0:
        v = -v
    basis = L.subquotient(L.complement(v))
    dual, a = L.dual_vec(v)
    if nrows(basis) != L.rank - 2:
        raise ValueError("Error constructing subquotient")
    M = Lattice(nrows(basis), L.batch_prod(basis, basis))
    A, T = M.lll(transform=True)
    basis = imat(list_rows(T @ basis))
    M = Lattice(M.rank, A)
    FPS = fp_search_cpp.FPSearch(np.array(-A, dtype=float), np.zeros(M.rank, dtype=float), 0, 2 * M.exp + 1)
    vecs = FPS.search_all()
    roots = []
    for u in vecs:
        if not M.is_root(u):
            continue
        w = imat(u) @ basis
        b = L.square(w)
        c, x, _, _, _ = euclid(2 * a, b)
        y = 2 * L.product(dual, w)
        if y % c != 0:
            continue
        w = w - (y // c) * x * v
        delta = abs(b // c)
        x, y = divmod(-L.product(w, base), (delta * L.product(v, base)))
        if y == 0:
            shift = [x - 1, x, x + 1]
        else:
            shift = [x, x + 1]
        for k in shift:
            roots.append(w + k * delta * v)
    return roots

L = I_lat(0, 2) + U_lat()
print(L.info())
v = imat([0, 0, 1, 0])
base = imat([0, 0, 1, 1])
roots = cusp_walls(L, v, base)
for r in roots:
    print(r, L.is_root(r))

Odd lattice of signature (1, 3), discriminant -1 and exponent 1
Discriminant group: [1, 1, 1, 1]
[1 1 -1 0] True
[1 1 0 0] True
[1 1 1 0] True
[0 1 -1 0] True
[0 1 0 0] True
[0 1 1 0] True
[-1 1 -1 0] True
[-1 1 0 0] True
[-1 1 1 0] True
[1 0 -1 0] True
[1 0 0 0] True
[1 0 1 0] True
[-1 0 -1 0] True
[-1 0 0 0] True
[-1 0 1 0] True
[1 -1 -1 0] True
[1 -1 0 0] True
[1 -1 1 0] True
[0 -1 -1 0] True
[0 -1 0 0] True
[0 -1 1 0] True
[-1 -1 -1 0] True
[-1 -1 0 0] True
[-1 -1 1 0] True


In [ ]:
L = I_lat(1, 21)
basis = L.even_sublattice()
L = Lattice(22, L.batch_prod(basis, basis))
print(L.info())
D = DiscForm(L)
iso = D.list_max_isospaces()
print(D.Ared, D.iso)

compl = [[[2, 0], [0, 6]], [[4, 2], [2, 4]]]
for b in compl:
    C = Lattice(2, b)
    print(C.info())
    D = DiscForm(L + C)
    print(D.Ared)
    iso = D.list_max_isospaces()
    for s in iso:
        M = D.overlattice(s)
        if M.parity == 0:
            print(M.info())
            DD = DiscForm(M)
            print(DD.iso)
            print(DD.Ared)

In [15]:
rank = 9
L = Lattice(rank, [[1 - 2 * int(i == j) for j in range(rank)] for i in range(rank)])
print(L.info())
V = Vinberg(L, base=[1] * rank, h_batch=20, fps_batch=10000)
walls = V.run(max_iterations=5)

Odd lattice of signature (1, 8), discriminant 1792 and exponent 14
Discriminant group: [1, 2, 2, 2, 2, 2, 2, 2, 14]
Iteration 5; 14 walls
Reached the maximum number of iterations


In [4]:
rays, lines = get_extremal_rays(walls, L.A)
print(len(rays), len(lines))
squares = [(ray * L.A_fl * ray.transpose())[0, 0] for ray in rays]
for r, s in zip(rays, squares):
    if s >= 0:
        continue
    ray, _ = flq2imat(r)
    print(ray, L.is_root(ray))
    adj = [w for w in walls if L.product(ray, w) == 0]
    for i in range(len(adj)):
        compl = L.complement([walls[j] for j in range(len(adj)) if i != j])
        if nrows(compl) != 2:
            continue
        M = L.batch_prod(compl, compl)
        Lcompl = BinLattice(M[0, 0], M[1, 1], M[0, 1])
        if Lcompl.zero:
            isotropic = imat(Lcompl.zero[0]) @ compl
            print(isotropic.tolist(), L.square(isotropic))

41 0
[10 -11 10 10 10 -11 10 10 3] False
[4, -5, 4, 4, 4, -5, 4, 4, 4] 0
[5, -4, -4, 5, -4, -4, -4, -4, -4] 0
[11 -10 11 11 11 -10 11 11 -3] False
[4, -5, 4, 4, 4, -5, 4, 4, 4] 0
[5, -4, -4, 5, -4, -4, -4, -4, -4] 0


In [17]:
v0 = imat([4, -5, 4, 4, 4, -5, 4, 4, 4])
walls = cusp_walls(L, v0, [1] * rank)
print(len(walls), walls)

106 [array([-3, 5, -4, -4, -5, 5, -4, -4, -4], dtype=object), array([1, 0, 0, 0, -1, 0, 0, 0, 0], dtype=object), array([5, -5, 4, 4, 3, -5, 4, 4, 4], dtype=object), array([-3, 5, -4, -4, -4, 5, -5, -4, -4], dtype=object), array([1, 0, 0, 0, 0, 0, -1, 0, 0], dtype=object), array([5, -5, 4, 4, 4, -5, 3, 4, 4], dtype=object), array([-3, 5, -4, -4, -4, 5, -4, -5, -4], dtype=object), array([1, 0, 0, 0, 0, 0, 0, -1, 0], dtype=object), array([5, -5, 4, 4, 4, -5, 4, 3, 4], dtype=object), array([-4, 5, -4, -5, -4, 5, -4, -3, -4], dtype=object), array([0, 0, 0, -1, 0, 0, 0, 1, 0], dtype=object), array([4, -5, 4, 3, 4, -5, 4, 5, 4], dtype=object), array([-4, 5, -4, -4, -5, 5, -4, -3, -4], dtype=object), array([0, 0, 0, 0, -1, 0, 0, 1, 0], dtype=object), array([4, -5, 4, 4, 3, -5, 4, 5, 4], dtype=object), array([-4, 5, -4, -4, -4, 5, -5, -3, -4], dtype=object), array([0, 0, 0, 0, 0, 0, -1, 1, 0], dtype=object), array([4, -5, 4, 4, 4, -5, 3, 5, 4], dtype=object), array([-4, 5, -4, -4, -4, 5, -3, -4

In [6]:
print(irred_decomp(M))

[[[0, 0, 0, 0, 0, 0, -1], [0, 0, 0, 0, 0, -1, 0], [0, 0, 0, -1, 0, 0, 0], [0, 0, -1, 0, 0, 0, 0], [0, -1, 0, 0, 0, 0, 0], [-28, -1, -1, -1, 11, -1, -1]], [[5, 0, 0, 0, -2, 0, 0]]]


In [4]:
L = D_lat(20)(-1) + U_lat()
print(L.info())
V = Vinberg(L, base=[0] * 20 + [1, 1], h_batch=1, fps_batch=10000)
walls = V.run(root_batch=10, max_iterations=3)
print(len(walls), 'walls found')
for w in walls:
    print(w, L.square(w))
rays, lines = get_extremal_rays(walls, L.A)
print(len(rays), len(lines))
squares = [(fl.fmpq_mat(1, L.rank, ray) * L.A * fl.fmpq_mat(L.rank, 1, ray))[0, 0] for ray in rays]
for r, s in zip(rays, squares):
    print(r, s)

Even lattice of signature (1, 21), discriminant -4 and exponent 2
Discriminant group: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2]
Iteration 3; 22 walls
Reached the maximum number of iterations
22 walls found
[0, 0, 0, 0, 0, 0, -1, -1, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -1, -1, 0, 0] -2
[0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 0, 0] -2
[0, 0, 0, 0, 0, -1, -1, -1, -1, -1, -1, -1, -2, -2, -2, -2, -2, -2, -1, -1, 0, 0] -2
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0] -2
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0] -2
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0] -2
[0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 0, 0] -2
[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0] -2
[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 0, 0] -2
[0, 0, 0, 0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -2, -1, -1, 0, 0] -2
[0, 0, 0, 0, 0, 0, 0, 0

In [5]:
for r, s in zip(rays, squares):
    if s >= 0:
        continue
    ray, _ = fl.fmpq_mat(1, L.rank, r).numer_denom()
    print(ray, L.is_root(ray))
    adj = [w for w in walls if L.product(ray, w) == 0]
    for i in range(len(adj)):
        compl = L.complement([walls[j] for j in range(len(adj)) if i != j])
        M = L.batch_prod(compl, compl)
        Lcompl = BinLattice(M[0][0], M[1][1], M[0][1])
        print(Lcompl.info())

[-1, -2, -1, -1, -1, -2, -2, -1, -1, 0, -1, -1, 0, -1, 0, -1, -1, -1, -1, 0, 2, 2] False
Even lattice of signature (1, 1) and discriminant -28
Does not represent zero
Canonical basis: (0, 1), (-1, 0)
Canonical form: 2, -14, 0
River length: 7

Even lattice of signature (1, 1) and discriminant -136
Does not represent zero
Canonical basis: (0, 1), (-1, 0)
Canonical form: 2, -68, 0
River length: 16

Even lattice of signature (1, 1) and discriminant -16
Represents zero at (1, 2), (-1, -2), (-1, 2), (1, -2)
Canonical basis: (0, 1), (1, -1)
Canonical form: 2, -6, -2
River length: 3

Even lattice of signature (1, 1) and discriminant -56
Does not represent zero
Canonical basis: (0, 1), (-1, 0)
Canonical form: 2, -28, 0
River length: 10

Even lattice of signature (1, 1) and discriminant -24
Does not represent zero
Canonical basis: (0, 1), (-1, 0)
Canonical form: 2, -12, 0
River length: 6

Even lattice of signature (1, 1) and discriminant -152
Does not represent zero
Canonical basis: (0, 1), (-1,